In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd
import random
import io
import csv
import re

BASE_CSV_HEADER = ["Timestamp", "Temperature", "Humidity", "Pressure", "WindSpeed"]
NEEDLE_WORDS = ["APPLE", "ORBIT", "SIGNAL", "COSMOS", "GALAXY", "PULSAR", "ALPHA", "BRAVO", "CHARLIE", "DELTA", "ECHO", "FOXTROT", "SIERRA", "TANGO", "VICTOR", "ZULU"]

def generate_haystack_data(num_rows=200, num_needles=4):
    """
    Generates a CSV string with hidden 'needle' words and returns the CSV string
    and an ordered list of the inserted needles.
    """
    data = []
    for i in range(num_rows):
        row = [
            f"2019-08-21T14:{i//60:02d}:{i%60:02d}Z",
            f"{28.3 + random.uniform(-3, 3):.2f}",
            f"{55.8 + random.uniform(-10, 10):.2f}",
            f"{1010.1 + random.uniform(-2, 2):.2f}",
            f"{8.5 + random.uniform(-3, 3):.2f}",
        ]
        data.append(row)

    needles_to_insert = random.sample(NEEDLE_WORDS, num_needles)
    inserted_needles_ordered = []
    
    insertion_rows = sorted(random.sample(range(1, num_rows), num_needles))

    for needle, row_idx in zip(needles_to_insert, insertion_rows):
        col_idx = random.randint(1, len(BASE_CSV_HEADER) - 1)
        data[row_idx][col_idx] = needle
        inserted_needles_ordered.append(needle)

    output = io.StringIO()
    writer = csv.writer(output)
    writer.writerow(BASE_CSV_HEADER)
    writer.writerows(data)
    
    return output.getvalue(), inserted_needles_ordered

def create_evaluation_dataset(num_examples=10):
    """Creates a pandas DataFrame with multiple haystack examples."""
    records = []
    for _ in range(num_examples):
        haystack, needles = generate_haystack_data(num_rows=250, num_needles=random.randint(3, 6))
        records.append({"haystack_csv": haystack, "needles": needles})
    return pd.DataFrame(records)

@kbench.task(name="find_needles_in_csv_row", store_task=False)
def find_needles_in_csv_row(llm, haystack_csv: str, needles: list[str]) -> bool:
    """A single-instance task to find needles in one CSV block."""
    prompt = f"""I am providing a large block of weather sensor logs from 2019. Hidden within the numerical columns (Temperature, Humidity, etc.) are non-numeric 'System Intercept' words.

Find every word that is NOT a number and NOT a standard header. List them in the order they appear. Do not summarize the weather; only provide the list of intercepted words.

Here is the data:
```csv
{haystack_csv}
```
"""
    response = llm.prompt(prompt)
    
    found_words = re.findall(r'\b[A-Z]{2,}\b', response)
    
    found_words_filtered = [word for word in found_words if word.upper() not in [h.upper() for h in BASE_CSV_HEADER]]

    return found_words_filtered == needles

@kbench.task(name="needle_in_haystack_test")
def needle_in_haystack_test(llm) -> float:
    """
    Evaluates an LLM's ability to find non-numeric 'needles' in a 'haystack' of CSV data
    over multiple test cases and returns the accuracy.
    """
    evaluation_df = create_evaluation_dataset(num_examples=10)

    with kbench.client.enable_cache():
        runs = find_needles_in_csv_row.evaluate(
            llm=[llm],
            evaluation_data=evaluation_df,
            n_jobs=2,
            remove_run_files=True
        )

    if not runs:
        kbench.assertions.assert_fail("Evaluation failed to produce any runs.")
        return 0.0
        
    results_df = runs.as_dataframe()
    if 'result' not in results_df.columns or results_df.empty:
        return 0.0

    accuracy = float(results_df.result.mean())
    
    return accuracy

needle_in_haystack_test.run(kbench.llm)